# Day 20 - Merge Sort, Quick Sort, and their parallel versions

Both are divide and conquer. The difference is **where the work happens**:

| | split | join |
|---|---|---|
| merge sort | trivial (cut in half) | all the work (the merge) |
| quick sort | all the work (the partition) | trivial (nothing) |

Everything else follows from that one line. Merge sort is stable, is always
`O(n log n)`, needs `O(n)` scratch space, and is what you use when the data does not
fit in memory. Quick sort sorts in place with a much smaller constant, and pays with
an `O(n^2)` worst case that a hostile - or merely *sorted* - input can trigger.

The last third asks the parallel question, and the answer is more interesting than
"just recurse on two cores".

In [1]:
import random
from bisect import bisect_left
from heapq import merge as heap_merge

## 1. Merge sort

The merge is a two-finger walk down two sorted lists. The `<=` is not cosmetic: it is
the entire reason merge sort is **stable**.

In [2]:
def merge(left, right, stats=None):
    """Merge two sorted lists.  `<=` is what makes the whole sort stable."""
    out = []
    i = j = 0
    while i < len(left) and j < len(right):
        if stats is not None:
            stats['cmp'] += 1
        if left[i] <= right[j]:            # <= keeps equal elements in order
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    out.extend(left[i:])
    out.extend(right[j:])
    return out


def merge_sort(a, stats=None, depth=0):
    if stats is not None:
        stats['depth'] = max(stats['depth'], depth)
    if len(a) <= 1:
        return list(a)
    mid = len(a) // 2
    left = merge_sort(a[:mid], stats, depth + 1)
    right = merge_sort(a[mid:], stats, depth + 1)
    return merge(left, right, stats)

In [3]:
small = [5, 2, 9, 1, 5, 6]
st = {'cmp': 0, 'depth': 0}
print('sorted     ', merge_sort(small, st))
print('comparisons', st['cmp'], ' depth', st['depth'])

pairs = [(1, 'a'), (0, 'b'), (1, 'c'), (0, 'd')]
print('stable     ', merge_sort(pairs), '<- a before c, b before d')

sorted      [1, 2, 5, 5, 6, 9]
comparisons 10  depth 3
stable      [(0, 'b'), (0, 'd'), (1, 'a'), (1, 'c')] <- a before c, b before d


### The free by-product: counting inversions

When the merge emits an element from the right half, that element has just jumped ahead
of *everything still unconsumed in the left half* - and each of those pairs is an
inversion. Counting them costs one addition per emit, so you get "how far is this list
from sorted?" for free.

In [4]:
def count_inversions(a):
    """Merge sort's free by-product: how far the input is from sorted.

    Every time an element of the right half is emitted, it jumps ahead of
    everything still left in the left half - and each of those is an inversion.
    """
    def go(a):
        if len(a) <= 1:
            return list(a), 0
        mid = len(a) // 2
        left, x = go(a[:mid])
        right, y = go(a[mid:])
        out, inv = [], x + y
        i = j = 0
        while i < len(left) and j < len(right):
            if left[i] <= right[j]:
                out.append(left[i]); i += 1
            else:
                inv += len(left) - i        # <- the whole trick
                out.append(right[j]); j += 1
        out.extend(left[i:]); out.extend(right[j:])
        return out, inv
    return go(a)[1]

In [5]:
print('inversions in [2, 4, 1, 3, 5] =', count_inversions([2, 4, 1, 3, 5]))
print('sorted   ->', count_inversions(list(range(20))))
print('reversed ->', count_inversions(list(range(20, 0, -1))), '= 20*19/2, the maximum')

inversions in [2, 4, 1, 3, 5] = 3
sorted   -> 0
reversed -> 190 = 20*19/2, the maximum


## 2. Quick sort

No merge step at all: partition the array so that everything small is on the left and
everything large is on the right, and the two halves are already in their final
regions.

Lomuto's partition is the one everybody writes first. `three_way_partition` is the
Dutch national flag: it produces three regions, and the middle one - the values equal
to the pivot - is **finished**, so neither recursive call ever looks at it again.

In [6]:
def quicksort_lomuto(a, stats=None, pivot='last', rng=None, lo=0, hi=None,
                     depth=0):
    """In-place quick sort, Lomuto partition, two-way.

    pivot='last'   textbook version - O(n^2) on already sorted input
    pivot='random' the one-line fix
    """
    if hi is None:
        hi = len(a) - 1
    if stats is not None:
        stats['depth'] = max(stats['depth'], depth)
    if lo >= hi:
        return a
    if pivot == 'random':
        k = rng.randint(lo, hi)
        a[k], a[hi] = a[hi], a[k]
    p = a[hi]
    i = lo
    for j in range(lo, hi):
        if stats is not None:
            stats['cmp'] += 1
        if a[j] < p:
            a[i], a[j] = a[j], a[i]
            i += 1
    a[i], a[hi] = a[hi], a[i]
    quicksort_lomuto(a, stats, pivot, rng, lo, i - 1, depth + 1)
    quicksort_lomuto(a, stats, pivot, rng, i + 1, hi, depth + 1)
    return a


def three_way_partition(a, lo, hi, p, stats=None):
    """Dutch national flag: everything equal to the pivot lands in the middle
    and is never looked at again.  This is the fix for duplicate-heavy data."""
    lt, i, gt = lo, lo, hi
    while i <= gt:
        if stats is not None:
            stats['cmp'] += 1
        if a[i] < p:
            a[lt], a[i] = a[i], a[lt]; lt += 1; i += 1
        elif a[i] > p:
            a[i], a[gt] = a[gt], a[i]; gt -= 1
        else:
            i += 1
    return lt, gt


def quicksort_3way(a, stats=None, rng=None, lo=0, hi=None, depth=0):
    if hi is None:
        hi = len(a) - 1
    if stats is not None:
        stats['depth'] = max(stats['depth'], depth)
    if lo >= hi:
        return a
    k = rng.randint(lo, hi) if rng else (lo + hi) // 2
    lt, gt = three_way_partition(a, lo, hi, a[k], stats)
    quicksort_3way(a, stats, rng, lo, lt - 1, depth + 1)
    quicksort_3way(a, stats, rng, gt + 1, hi, depth + 1)
    return a


def new_stats():
    return {'cmp': 0, 'depth': 0}

In [7]:
demo = [7, 2, 9, 4, 7, 1, 7, 3]
b = list(demo)
lt, gt = three_way_partition(b, 0, len(b) - 1, 7)
print('pivot 7 ->', b[:lt], b[lt:gt + 1], b[gt + 1:], '   <7 | ==7 | >7')

a = list(demo); st = new_stats(); quicksort_lomuto(a, st, 'last')
print('lomuto   ', a, st['cmp'], 'comparisons')
a = list(demo); st = new_stats(); quicksort_3way(a, st, random.Random(1))
print('three-way', a, st['cmp'], 'comparisons')

pivot 7 -> [2, 3, 4, 1] [7, 7, 7] [9]    <7 | ==7 | >7
lomuto    [1, 2, 3, 4, 7, 7, 7, 9] 16 comparisons
three-way [1, 2, 3, 4, 7, 7, 7, 9] 26 comparisons


## 3. The two inputs that break the textbook version

A pivot that always lands at one end means one element is removed per level, so the
recursion is `n` deep and the comparisons are `n^2/2`.

* **already sorted** + last-element pivot: the classic. A random pivot fixes it.
* **all equal**: a random pivot does *not* help - every two-way partition still splits
  `k` equal values into `k-1` and `0`. Only the three-way split fixes this, because it
  retires the equal block instead of re-partitioning it.

In [8]:
n = 200
rng = random.Random(7)
print('n = 200        lomuto/last  lomuto/random   three-way   merge sort')
for name, arr in (('already sorted', list(range(n))),
                  ('all equal     ', [7] * n),
                  ('random        ', [rng.randint(0, 999) for _ in range(n)])):
    r = []
    for f in (lambda x: quicksort_lomuto(x, s, 'last'),
              lambda x: quicksort_lomuto(x, s, 'random', random.Random(7)),
              lambda x: quicksort_3way(x, s, random.Random(7)),
              lambda x: merge_sort(x, s)):
        s = new_stats(); f(list(arr)); r.append(s)
    print('%s  cmp %7d %10d %12d %12d' % (name, r[0]['cmp'], r[1]['cmp'],
                                          r[2]['cmp'], r[3]['cmp']))
    print('%s  dep %7d %10d %12d %12d' % (' ' * 14, r[0]['depth'],
                                          r[1]['depth'], r[2]['depth'],
                                          r[3]['depth']))

n = 200        lomuto/last  lomuto/random   three-way   merge sort
already sorted  cmp   19900       1502         1494          732
                dep     199         15           12            8
all equal       cmp   19900      19900          200          732
                dep     199        199            1            8
random          cmp    1456       1484         1685         1299
                dep      14         12           15            8


## 4. External sort - when the data does not fit in memory

This is merge sort's other superpower, and the reason it never went away. Read as much
as fits, sort it, spill it as a *run*; then merge all the runs with a heap that holds
one element per run. Peak memory is the chunk size plus the number of runs - not `n`.

Quick sort cannot do this: partitioning needs random access to the whole array.

In [9]:
def external_sort(data, chunk):
    """Sort more data than fits in memory.

    Read `chunk` items at a time, sort that much in RAM, spill it to a run,
    then merge all the runs with a heap - which only ever holds one element
    per run.  Peak memory is chunk + number of runs, not n.
    """
    runs = [sorted(data[i:i + chunk]) for i in range(0, len(data), chunk)]
    peak = max(chunk, len(runs))
    return list(heap_merge(*runs)), len(runs), peak

In [10]:
rng = random.Random(20)
big = [rng.randint(0, 9999) for _ in range(1000)]
out, runs, peak = external_sort(big, chunk=100)
print('%d values, %d in memory at a time -> %d runs' % (len(big), 100, runs))
print('peak items resident: %d (%.0f%% of the data), correct: %s'
      % (peak, 100 * peak / len(big), out == sorted(big)))

1000 values, 100 in memory at a time -> 10 runs
peak items resident: 100 (10% of the data), correct: True


## 5. Parallel: work vs depth

Two numbers describe a parallel algorithm:

* **work** = total operations (what one core would do),
* **depth** = the longest chain of dependent operations (what infinitely many cores
  would still have to wait for).

`work / depth` is the most parallelism the algorithm can ever use. Recursing on the two
halves in parallel *looks* like the whole answer, but it is not: the final merge of `n`
elements is one sequential loop, so the depth stays `O(n)` and the parallelism stays
around 5x no matter how many cores you buy.

The fix is a merge that is itself divide and conquer: take the median of the longer
half, binary-search for its position in the other half, and the two sides of that split
can be merged independently.

In [11]:
WORKDEPTH_N = 1024


def merge_seq(a, b):
    """The ordinary merge: n steps, and every one of them waits for the last."""
    out = merge(a, b)
    return out, len(a) + len(b), len(a) + len(b)


def merge_par(a, b):
    """Divide-and-conquer merge.

    Take the median of the longer list, binary-search for its place in the
    other, and the two sides of that split can be merged independently.  The
    work barely changes; the depth collapses from n to log^2 n.
    """
    if len(a) < len(b):
        a, b = b, a
    if not a:
        return list(b), max(len(b), 1), 1
    if not b:
        return list(a), len(a), 1
    if len(a) == 1:
        return merge(a, b), 2, 1
    mid = len(a) // 2
    piv = a[mid]
    cost = max(1, len(b).bit_length())          # the binary search
    j = bisect_left(b, piv)
    left, wl, dl = merge_par(a[:mid], b[:j])
    right, wr, dr = merge_par(a[mid + 1:], b[j:])
    return left + [piv] + right, wl + wr + cost + 1, max(dl, dr) + cost


def msort_cost(a, merger, parallel_recursion):
    """Run merge sort while accumulating (work, depth).

    Sequential composition adds depths; parallel composition takes the max.
    That single choice is the whole difference between the three rows of the
    table this prints.
    """
    if len(a) <= 1:
        return list(a), 0, 0
    mid = len(a) // 2
    l, wl, dl = msort_cost(a[:mid], merger, parallel_recursion)
    r, wr, dr = msort_cost(a[mid:], merger, parallel_recursion)
    out, wm, dm = merger(l, r)
    below = max(dl, dr) if parallel_recursion else dl + dr
    return out, wl + wr + wm, below + dm


def workdepth_input(m=WORKDEPTH_N):
    """A fixed permutation of 0..m-1.

    Work and depth depend on the actual values, so the script, the figure and
    the demo page all measure this same array - otherwise the three artefacts
    would quote three slightly different numbers for the same claim.
    """
    return [(i * 37) % m for i in range(m)]

In [12]:
m = 1024
arr = workdepth_input(m)   # a fixed permutation, so the article, the figure
#                            and the demo page all measure the same array
print('n = %d' % m)
print('%-32s %10s %8s %12s' % ('', 'work', 'depth', 'parallelism'))
for name, mg, par in (('sequential', merge_seq, False),
                      ('parallel recursion, seq merge', merge_seq, True),
                      ('parallel recursion, par merge', merge_par, True)):
    out, w, d = msort_cost(arr, mg, par)
    assert out == sorted(arr)
    print('%-32s %10d %8d %11.1fx' % (name, w, d, w / d))

n = 1024
                                       work    depth  parallelism
sequential                            10240    10240         1.0x
parallel recursion, seq merge         10240     2046         5.0x
parallel recursion, par merge         18857      235        80.2x


## 6. Sample sort - the same idea across machines

On a cluster you cannot afford `log n` rounds of pairwise merging. Sample sort does the
split *once*: sample the data, sort the samples, take `p-1` splitters, send every value
to the machine that owns its range, and let each machine sort locally. One all-to-all
exchange, then embarrassingly parallel.

The only thing that can go wrong is load imbalance, and the whole sort finishes when
the **slowest** bucket finishes. Oversampling is how you buy that down.

In [13]:
def sample_sort(data, p, oversample, rng):
    """How a sort is done across p machines.

    Pick p*oversample samples, sort them, take every oversample-th as a
    splitter, and every machine keeps the bucket it owns.  One all-to-all
    exchange, then each machine sorts locally.  The only thing that can go
    wrong is load imbalance - which is exactly what oversampling buys down.
    """
    s = rng.sample(data, min(len(data), p * oversample))
    s.sort()
    splitters = [s[i * oversample] for i in range(1, p)]
    buckets = [[] for _ in range(p)]
    for x in data:
        buckets[bisect_left(splitters, x)].append(x)
    out = []
    for b in buckets:
        out.extend(sorted(b))                    # each machine, in parallel
    sizes = [len(b) for b in buckets]
    return out, sizes, max(sizes) / (len(data) / p)

In [14]:
prng = random.Random(99)
payload = [prng.randint(0, 10 ** 6) for _ in range(20000)]

# sampling is a lottery, so average each row over 200 different samples
print('oversample   mean imbalance   worst seen')
for over in (1, 4, 32):
    imbs = [sample_sort(payload, 8, over, random.Random(t))[2]
            for t in range(200)]
    print('%10d %14.2fx %11.2fx' % (over, sum(imbs) / len(imbs), max(imbs)))

print()
print('one run, oversample  1:', sample_sort(payload, 8, 1, random.Random(5))[1])
print('the same, oversample 32:', sample_sort(payload, 8, 32, random.Random(5))[1])

oversample   mean imbalance   worst seen
         1           2.76x        5.54x
         4           1.82x        3.41x
        32           1.26x        1.82x

one run, oversample  1: [4019, 3870, 1069, 1315, 992, 4552, 341, 3842]
the same, oversample 32: [2872, 3081, 2792, 2232, 2475, 2331, 1765, 2452]


## 7. LeetCode 912 - Sort an Array

"Sort without using the built-in sort" is a quick sort test in disguise, and the judge's
data contains exactly the two killers above. The accepted shape is: random pivot,
three-way partition, insertion sort for short runs, and **recurse on the smaller side
while looping on the larger** so the call stack stays `O(log n)`.

In [15]:
def sort_an_array(nums):
    """LC 912 asks you to sort without the library sort - it is a quick sort
    test in disguise, and the test data contains long runs of equal values and
    sorted stretches, precisely the two inputs the textbook version dies on."""
    rng = random.Random(912)
    a = list(nums)

    def go(lo, hi):
        while lo < hi:
            if hi - lo < 16:                     # small runs: insertion sort
                for i in range(lo + 1, hi + 1):
                    v = a[i]; j = i - 1
                    while j >= lo and a[j] > v:
                        a[j + 1] = a[j]; j -= 1
                    a[j + 1] = v
                return
            k = rng.randint(lo, hi)
            lt, gt = three_way_partition(a, lo, hi, a[k])
            if lt - lo < hi - gt:                # recurse on the smaller side,
                go(lo, lt - 1)                   # loop on the larger one:
                lo = gt + 1                      # depth stays O(log n)
            else:
                go(gt + 1, hi)
                hi = lt - 1
    go(0, len(a) - 1)
    return a

In [16]:
print(sort_an_array([5, 2, 3, 1]))
print(sort_an_array([5, 1, 1, 2, 0, 0]))

st = new_stats(); quicksort_lomuto([7] * 400, st, 'last')
print('400 equal values, textbook:', st['cmp'], 'comparisons, depth', st['depth'])
st = new_stats(); quicksort_3way([7] * 400, st, random.Random(3))
print('400 equal values, three-way:', st['cmp'], 'comparisons, depth', st['depth'])

[1, 2, 3, 5]
[0, 0, 1, 1, 2, 5]
400 equal values, textbook: 79800 comparisons, depth 399
400 equal values, three-way: 400 comparisons, depth 1


## Tests

In [17]:
for trial in range(60):
    r = random.Random(trial)
    arr = [r.randint(0, 20) for _ in range(r.randint(0, 40))]
    exp = sorted(arr)
    assert merge_sort(arr) == exp
    assert quicksort_lomuto(list(arr), None, 'random', r) == exp
    assert quicksort_3way(list(arr), None, random.Random(trial)) == exp
    assert sort_an_array(arr) == exp
    assert external_sort(arr, 7)[0] == exp
    assert msort_cost(arr, merge_par, True)[0] == exp
    assert count_inversions(arr) == sum(
        1 for i in range(len(arr)) for j in range(i + 1, len(arr))
        if arr[i] > arr[j])
print('all assertions passed')

all assertions passed
